# Evaluate a Hugging Face model on six benchmarks

This notebook runs **MMLU, GSM8K, GPQA, HumanEval, TruthfulQA, and IFEval** using the
EleutherAI LM Evaluation Harness. The default model is the **base**
`meta-llama/Llama-3.2-1B`. Run cells from top to bottom in a fresh Python 3.10–3.12
kernel. Keep this notebook beside `evaluation.py` and `requirements.txt`.

A Linux GPU runtime is recommended. Full evaluation can take hours; select
`SMOKE_TEST=True` for a two-example-per-task plumbing check. A smoke test still
includes all MMLU subjects and is **not a reportable full benchmark result**.

HumanEval runs generated Python. Use a disposable, isolated runtime. Its upstream
execution limits are not a sandbox. Code execution is enabled below so the default
suite includes all six requested benchmarks.

See [README.md](README.md) for inputs, protocol details, and troubleshooting.

## 1. Inputs

Provide a Hub model ID, root model-card URL, or local `save_pretrained` directory.
The harness input can be a Git URL or an existing checkout. For an existing
checkout at its current commit, set `HARNESS_REF=None`.

The base model uses raw completion prompts. Enable `APPLY_CHAT_TEMPLATE` for an
instruction-tuned model with a matching chat tokenizer. HumanEval retains its raw
function-completion prompt regardless of this option.

In [1]:
from pathlib import Path
import os
import sys

# Locate the implementation whether Jupyter starts here or at the repository root.
candidates = [Path.cwd(), Path.cwd() / "notebooks/evaluation/lm_harness"]
EVAL_DIR = next((path.resolve() for path in candidates if (path / "evaluation.py").is_file()), None)
if EVAL_DIR is None:
    raise FileNotFoundError("Open Jupyter in this folder or the repository root, keeping evaluation.py beside the notebook.")
sys.path.insert(0, str(EVAL_DIR))

MODEL_SOURCE = "meta-llama/Llama-3.2-1B"
# MODEL_SOURCE = "https://huggingface.co/meta-llama/Llama-3.2-1B"
# MODEL_SOURCE = "/absolute/path/to/my-hf-checkpoint"
MODEL_REVISION = "main"
TOKENIZER_SOURCE = None  # None means the tokenizer supplied with MODEL_SOURCE.

HARNESS_SOURCE = "https://github.com/EleutherAI/lm-evaluation-harness.git"
HARNESS_REF = "v0.4.11"  # Set None to use the current commit of a local checkout.
HARNESS_DESTINATION = EVAL_DIR / ".cache/lm-evaluation-harness"
INSTALL_DEPENDENCIES = True  # Set False after successful setup + kernel restart.

SELECTED_BENCHMARKS = ["MMLU", "GSM8K", "GPQA", "HumanEval", "TruthfulQA", "IFEval"]
SMOKE_TEST = True
SMOKE_EXAMPLES_PER_TASK = 2

DEVICE = "auto"  # auto -> CUDA if available, otherwise CPU; or require "cuda:0".
DTYPE = "auto"   # Native BF16 on Ampere+; FP16 on older CUDA GPUs; CPU FP32.
BATCH_SIZE = "auto"   # Increase with sufficient VRAM, or use "auto" (capped at 8).
MAX_LENGTH = 4096  # Context cap; increase if long prompts are truncated.
APPLY_CHAT_TEMPLATE = False
TRUST_REMOTE_CODE = False
ALLOW_HUMANEVAL_EXECUTION = True
LOG_SAMPLES = True
OUTPUT_ROOT = EVAL_DIR / "artifacts"

# Optional: supply an ALREADY LOADED transformers.PreTrainedModel and tokenizer.
# Its existing weights/device/dtype take precedence over MODEL_SOURCE/DEVICE/DTYPE.
LOADED_MODEL = None
LOADED_TOKENIZER = None

# Keep new caches inside this implementation folder. Honor existing cache settings.
os.environ.setdefault("HF_HOME", str(EVAL_DIR / ".cache/huggingface"))
os.environ.setdefault("NLTK_DATA", str(EVAL_DIR / ".cache/nltk_data"))
Path(os.environ["NLTK_DATA"]).mkdir(parents=True, exist_ok=True)
print("Evaluation folder:", EVAL_DIR)

Evaluation folder: /home/e12056/ASI/VP-DPO/notebooks/evaluation/lm_harness


## 2. Install the harness and dependencies

The default release is verified against a fixed commit. Existing checkouts are
used unchanged. Installation uses the current notebook kernel's Python.

If you already imported Torch, Transformers, datasets, or lm_eval before
installation, restart the kernel afterward, set `INSTALL_DEPENDENCIES=False`,
and rerun. The source-path check below catches importing the wrong harness.

In [2]:
from evaluation import prepare_harness

harness_dir = prepare_harness(
    HARNESS_DESTINATION,
    source=HARNESS_SOURCE,
    ref=HARNESS_REF,
    install=INSTALL_DEPENDENCIES,
)

Obtaining file:///home/e12056/ASI/VP-DPO/notebooks/evaluation/lm_harness/.cache/lm-evaluation-harness
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lm_eval (pyproject.toml): started
  Building editable for lm_eval (pyproject.toml): finished with status 'done'
  Created wheel for lm_eval: filename=lm_eval-0.4.11-0.editable-py3-none-any.whl size=28546 sha256=46f46ed23d3c3ac9ddb033c168f1fe0ee6e7fe65a9bd1be004c1b0f6d02b6f36
  Stored in directory: /tmp/pip-ephem-wheel-cache-zhrt_cyc/wheels/d3/c9/87/bed1219ff63b3756286f7ecc09f

## 3. Authenticate and configure

Accept/request access to [Llama 3.2 1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
and [GPQA](https://huggingface.co/datasets/Idavidrein/gpqa) first. This cell uses an
existing Hugging Face token or asks through a hidden prompt. It does not write a
token into notebook source or output. A blank token is allowed for public/local
inputs; gated-resource checks will explain access failures.

In [3]:
from getpass import getpass
from huggingface_hub import get_token
from evaluation import BENCHMARKS, EvalConfig, normalize_model_source, resolve_device_dtype
import pandas as pd

if not get_token():
    entered_token = getpass("HF read token (hidden; Enter to use public/local resources only): ").strip()
    if entered_token:
        os.environ["HF_TOKEN"] = entered_token
    del entered_token

config = EvalConfig(
    model=normalize_model_source(MODEL_SOURCE) if LOADED_MODEL is None else str(MODEL_SOURCE),
    revision=MODEL_REVISION,
    tokenizer=TOKENIZER_SOURCE,
    benchmarks=tuple(SELECTED_BENCHMARKS),
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    limit=SMOKE_EXAMPLES_PER_TASK if SMOKE_TEST else None,
    apply_chat_template=APPLY_CHAT_TEMPLATE,
    trust_remote_code=TRUST_REMOTE_CODE,
    allow_humaneval_execution=ALLOW_HUMANEVAL_EXECUTION,
    log_samples=LOG_SAMPLES,
)
config.validate()
print("Mode:", "SMOKE — limited examples per task" if SMOKE_TEST else "FULL")
if LOADED_MODEL is None:
    print("Model:", config.model)
    print("Device/dtype:", resolve_device_dtype(DEVICE, DTYPE))
else:
    print("Loaded model:", type(LOADED_MODEL).__name__, LOADED_MODEL.device, LOADED_MODEL.dtype)
display(pd.DataFrame([
    {"benchmark": name, "tasks": ", ".join(BENCHMARKS[name].tasks),
     "fewshot": BENCHMARKS[name].fewshot, "metrics": BENCHMARKS[name].description}
    for name in config.benchmarks
]))

/home/e12056/ASI/VP-DPO/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mode: SMOKE — limited examples per task
Model: meta-llama/Llama-3.2-1B
Device/dtype: ('cuda:0', 'float16')


,benchmark,tasks,fewshot,metrics
0,MMLU,mmlu,5,All 57 subjects; accuracy
1,GSM8K,gsm8k,5,Strict and flexible exact match
2,GPQA,gpqa_main_zeroshot,0,Main split; acc and acc_norm
3,HumanEval,humaneval,0,Greedy code completion; pass@1
4,TruthfulQA,"truthfulqa_mc1, truthfulqa_mc2",0,MC1 and MC2
5,IFEval,ifeval,0,Prompt/instruction strict and loose accuracy


## 4. Preflight downloads and task checks

Check that all selected tasks exist, resolve a remote model revision to its commit,
check gated model access, cache GPQA Main, and prepare IFEval's NLTK tokenizer and
HumanEval's code metric. The evaluator downloads other datasets when needed.

GPQA uses Main zero-shot. TruthfulQA includes MC1 and MC2, with its built-in fixed
prompt examples and no additional few-shot sampling. Generation/scoring settings
come from the selected harness release. The notebook does not claim to reproduce
Meta's model-card scores.

In [4]:
from evaluation import preflight

preflight_info = preflight(config, harness_dir, loaded_model=LOADED_MODEL)
print(preflight_info)

Generating train split: 100%|██████████| 448/448 [00:00<00:00, 7901.83 examples/s]
/home/e12056/ASI/VP-DPO/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/e12056/ASI/VP-DPO/notebooks/evaluation/lm_harness/.cache/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):

{'tasks': ['mmlu', 'gsm8k', 'gpqa_main_zeroshot', 'humaneval', 'truthfulqa_mc1', 'truthfulqa_mc2', 'ifeval'], 'resolved_model_revision': '4e20de362430cd3b72f300e6b0f18e50e7166e08', 'gpqa_rows': 448}


## 5. Run evaluation

This is the compute-intensive cell. It loads the model once and saves each
benchmark before continuing. Full and smoke runs receive separate timestamped
directories. On failure, the exception remains visible and prior results stay on
disk. To retry a subset, edit `SELECTED_BENCHMARKS` and rerun from configuration.

For a custom loaded model, set `LOADED_MODEL` and `LOADED_TOKENIZER` in the inputs
cell. For a local checkpoint, save both model and tokenizer with
`save_pretrained` and use that directory as `MODEL_SOURCE`.

In [ ]:
from evaluation import run_suite

run_dir = run_suite(
    config,
    harness_dir=harness_dir,
    output_root=OUTPUT_ROOT,
    loaded_model=LOADED_MODEL,
    tokenizer=LOADED_TOKENIZER,
)
print("Completed. Results directory:", run_dir)

Artifacts: /home/e12056/ASI/VP-DPO/notebooks/evaluation/lm_harness/artifacts/20260905T183443Z_smoke_3c10bc99

MMLU: mmlu (5-shot)


Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 3001.94 examples/s]


## 6. Inspect scores and artifacts

Values are kept on the harness's native scale (generally 0–1). Both GSM8K
extraction filters, both TruthfulQA metrics, and all four IFEval metrics are
retained. MMLU's displayed aggregate is the harness's own weighted aggregate.
The CSV also contains every subject and subgroup. No cross-benchmark average is
computed.

`manifest.json` records settings, revision, dependencies, and run status.
`<benchmark>/results.json` retains task configurations and metrics.
`samples_*.jsonl` contains individual prompts, completions, and scores when enabled.

In [ ]:
import json

manifest = json.loads((run_dir / "manifest.json").read_text())
scores = pd.read_csv(run_dir / "summary.csv")
display(scores[(scores["benchmark"] != "MMLU") | (scores["task"] == "mmlu")].reset_index(drop=True))
print("Status:", manifest["status"])
print("Limited run:", manifest["limited_run"])
print("Harness commit:", manifest["harness_commit"])
print("All scores:", run_dir / "summary.csv")
print("Manifest:", run_dir / "manifest.json")

## 7. Reload an earlier run (optional)

Set `saved_run` to a timestamped result directory to inspect scores later. A failed
run may have a partial CSV; check its manifest for failed or pending benchmarks.

In [ ]:
# saved_run = Path("/absolute/path/to/artifacts/<run-id>")
# saved_manifest = json.loads((saved_run / "manifest.json").read_text())
# print(saved_manifest["status"], saved_manifest["benchmarks"])
# display(pd.read_csv(saved_run / "summary.csv"))